# cfdi-pandas Tutorial

Este notebook muestra como instalar y usar `cfdi-pandas` paso a paso con XMLs de CFDI.

## 1) Instalacion desde PyPI

Ejecuta esta celda al inicio del notebook.

In [1]:
# Instala la libreria desde PyPI en el entorno del notebook.
%pip install cfdi-pandas

Note: you may need to restart the kernel to use updated packages.


## 2) Importar libreria y cargar datos

Este ejemplo usa la carpeta `cfdi_data`. Si no existe, usa `cfdis_test`.

In [2]:
# Importa los metodos publicos de la libreria.
from pathlib import Path

from cfdi_pandas import (
    calculate_taxes,
    check_date_range,
    check_duplicate_uuid,
    check_tax_math,
    detect_cancelled,
    group_by_month,
    group_by_regimen,
    group_by_rfc,
    monthly_summary,
    read_cfdi,
    read_cfdi_folder,
    top_n,
    validate_all,
)

# Define la carpeta de datos principal y fallback.
base = Path.cwd()
data_folder = base / "cfdi_data"
if not data_folder.exists():
    data_folder = base / "cfdis_test"

# Lee todos los XML y separa DataFrames.
data = read_cfdi_folder(data_folder)
comprobantes = data["comprobantes"]
concepts = data["conceptos"]
taxes = data["impuestos"]

# Muestra conteos base para validar carga.
print("Data folder:", data_folder)
print("Invoices:", len(comprobantes))
print("Concept rows:", len(concepts))
print("Tax rows:", len(taxes))

Data folder: /app/cfdis_test
Invoices: 13
Concept rows: 14
Tax rows: 13


## 3) Ver estructura de datos

`read_cfdi_folder` regresa tres DataFrames: `comprobantes`, `conceptos` e `impuestos`.

In [3]:
# Muestra las primeras filas del DataFrame principal de facturas.
comprobantes.head()

,version,series,folio_number,issue_date,subtotal,total,invoice_type,currency,exchange_rate,place_of_issue,...,issuer_tax_regime,receiver_rfc,receiver_name,receiver_cfdi_use,receiver_fiscal_address,receiver_tax_regime,uuid,stamp_date,sat_certificate_number,cfd_seal
0,4.0,PA,122,2025-05-28T16:28:53,0,0,P,XXX,None,58187,...,626,VEN230503548,VENTRAUREN,CP01,11590,601,e5bcaecc-3fcd-4380-a8c4-d9552d14281c,2025-05-28T16:32:53,00001000000509846663,eup7IxzHZrNWdwBPzq2oDYaG7aFyz2kO9GO/qr9mVJD0t+...
1,4.0,A,1,2025-02-09T00:00:00,15.00,17.40,I,MXN,None,11590,...,601,RENJ0112285T3,JESUS REYNA NUÑEZ,S01,62577,605,7392dfdb-cc77-4d47-b3f2-a2840f2af633,2025-02-09T23:05:48,00001000000509846663,N6Pzryxiu+XDIQNIzhG8L1bg40hGfsp92qvAlbE9rxzjw6...
2,4.0,A,6,2025-08-05T00:00:00,50.00,50.00,I,MXN,None,01590,...,601,SAME030423JJ1,EMILIO SAID MACCISE,S01,52779,605,553856ff-c389-4002-b361-4d3f70aa8d32,2025-08-05T10:40:25,00001000000509846663,ICDSyis7GQzvoJTY7j09Ky3rXOGM7f8IZZ+r4AzDSu0y7z...
3,4.0,A,187,2025-04-28T19:01:22,390.50,452.98,I,MXN,None,66197,...,601,GAGJ010522MN7,JAVIER GARZA GUERRA,CP01,66266,605,36d47f4f-f779-4429-ba3d-e006efe72a15,2025-04-28T19:01:22,00001000000509846663,hpjVV/7xblIGKfr+ijNufD+S7gImsSQvWjW2JY1g0TAQy8...
4,4.0,NaN,19,2025-02-24T10:04:51,12.00,12.00,I,MXN,None,66197,...,601,RENJ0112285T3,JESUS REYNA NUÑEZ,S01,62577,605,85b29015-630d-477e-9f34-ed5e878c60d5,2025-02-24T10:04:51,00001000000509846663,Na6QorWnAJGTgPRZxsJHXwVUPDdo64mlPmB483mgmVrZsX...


In [4]:
# Muestra las primeras filas de conceptos por factura.
concepts.head()

,concept_index,product_service_key,description,quantity,unit_key,unit,unit_price,amount,discount,transferred_taxes_amount,withheld_taxes_amount,vat_amount,ieps_amount,concept_taxes,uuid
0,0,84111506,Pago,1,ACT,NaN,0,0,None,0,0,0,0,[],e5bcaecc-3fcd-4380-a8c4-d9552d14281c
1,0,84111506,Servicios de facturación,1.0,E48,NaN,5.0,5.0,None,0.8,0,0.8,0,"[{'tax_type': 'transfer', 'tax_code': '002', '...",7392dfdb-cc77-4d47-b3f2-a2840f2af633
2,1,81112500,Servicios de alquiler o arrendamiento de licen...,1.0,E48,NaN,10.0,10.0,None,1.6,0,1.6,0,"[{'tax_type': 'transfer', 'tax_code': '002', '...",7392dfdb-cc77-4d47-b3f2-a2840f2af633
3,0,84111506,Servicios de facturación,1,E48,NaN,50.0,50.0,None,0,0,0,0,[],553856ff-c389-4002-b361-4d3f70aa8d32
4,0,84111506,Servicio de facturación,1,E48,NaN,390.5,390.5,None,62.48,0,62.48,0,"[{'tax_type': 'transfer', 'tax_code': '002', '...",36d47f4f-f779-4429-ba3d-e006efe72a15


In [5]:
# Muestra las primeras filas de impuestos a nivel comprobante.
taxes.head()

,tax_type,tax_code,factor_type,rate_or_fee,base,amount,total_transferred_taxes,total_withheld_taxes,uuid
0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,e5bcaecc-3fcd-4380-a8c4-d9552d14281c
1,transfer,002,Tasa,0.160000,15.00,2.40,2.40,0,7392dfdb-cc77-4d47-b3f2-a2840f2af633
2,NaN,NaN,NaN,NaN,NaN,NaN,0,0,553856ff-c389-4002-b361-4d3f70aa8d32
3,transfer,002,Tasa,0.160000,390.50,62.48,62.48,0,36d47f4f-f779-4429-ba3d-e006efe72a15
4,NaN,NaN,NaN,NaN,NaN,NaN,0,0,85b29015-630d-477e-9f34-ed5e878c60d5


## 4) Analisis

Ejemplos de agregaciones y reportes basicos.

In [6]:
# Agrupa por RFC emisor y calcula total, conteo y promedio.
group_by_rfc(comprobantes, by="emisor")

,rfc,total_facturado,facturas,promedio_factura
0,AMU070829TW4,404.84,1,404.840000
1,NSI2104132IA,2024.40,3,674.800000
2,VEN230503548,2862.38,9,318.042222


In [7]:
# Resume facturacion por mes.
group_by_month(comprobantes)

,month,total_facturado,facturas
0,2025-02,1315.40,5
1,2025-03,0.00,1
2,2025-04,452.98,1
3,2025-05,1044.00,2
4,2025-06,0.00,1
5,2025-07,2429.24,2
6,2025-08,50.00,1


In [8]:
# Agrupa por regimen fiscal del emisor.
group_by_regimen(comprobantes)

,tax_regime,total_facturado,facturas
0,601,3267.22,10
1,626,2024.40,3


In [9]:
# Calcula resumen mensual de ingresos, egresos y neto.
monthly_summary(comprobantes)

,month,ingresos,egresos,neto
0,2025-02,1315.40,0.0,1315.40
1,2025-04,452.98,0.0,452.98
2,2025-07,404.84,2024.4,-1619.56
3,2025-08,50.00,0.0,50.00
4,2025-05,0.00,1044.0,-1044.00


In [10]:
# Muestra top de receptores por monto facturado.
top_n(comprobantes, by="receptor", n=10)

,rfc,total_facturado
0,VEN230503548,2429.24
1,DUVA590608U72,1276.00
2,BME111107JL3,1044.00
3,GAGJ010522MN7,462.98
4,SAME030423JJ1,50.00
5,RENJ0112285T3,29.40
6,LOLE040413AQA,0.00
7,MAN210826553,0.00


In [11]:
# Calcula IVA/IEPS/retenciones desde la tabla de impuestos globales.
calculate_taxes(taxes)

,vat_total,ieps_total,withholdings_total
0,719.95,0.0,0.0


In [12]:
# Calcula IVA/IEPS/retenciones desde impuestos anidados en conceptos.
calculate_taxes(concepts)

,vat_total,ieps_total,withholdings_total
0,719.9472,0.0,0.0


In [13]:
# Detecta facturas canceladas usando la columna status (si existe en tus datos).
cancelled = detect_cancelled(comprobantes)
print("Cancelled rows:", len(cancelled))
cancelled.head()

Cancelled rows: 0


,version,series,folio_number,issue_date,subtotal,total,invoice_type,currency,exchange_rate,place_of_issue,...,issuer_tax_regime,receiver_rfc,receiver_name,receiver_cfdi_use,receiver_fiscal_address,receiver_tax_regime,uuid,stamp_date,sat_certificate_number,cfd_seal


## 5) Validaciones

Chequeos de calidad de datos para UUID, calculos y rango de fechas.

In [14]:
# Detecta UUIDs duplicados entre facturas.
duplicates = check_duplicate_uuid(comprobantes)
print("Duplicate UUID rows:", len(duplicates))
duplicates

Duplicate UUID rows: 0


,uuid,count


In [15]:
# Valida que subtotal reporte coincida con suma de conceptos.
tax_math_issues = check_tax_math(comprobantes, concepts)
print("Tax math issue rows:", len(tax_math_issues))
tax_math_issues

Tax math issue rows: 0


,uuid,subtotal_reported,subtotal_calculated,difference


In [16]:
# Detecta CFDIs fuera de un rango de fechas mas estricto para ver resultados.
out_of_range = check_date_range(comprobantes, "2025-07-01", "2025-07-31")
print("Out-of-range rows:", len(out_of_range))
out_of_range.head()

Out-of-range rows: 11


,version,series,folio_number,issue_date,subtotal,total,invoice_type,currency,exchange_rate,place_of_issue,...,issuer_tax_regime,receiver_rfc,receiver_name,receiver_cfdi_use,receiver_fiscal_address,receiver_tax_regime,uuid,stamp_date,sat_certificate_number,cfd_seal
0,4.0,PA,122,2025-05-28 16:28:53,0,0,P,XXX,None,58187,...,626,VEN230503548,VENTRAUREN,CP01,11590,601,e5bcaecc-3fcd-4380-a8c4-d9552d14281c,2025-05-28T16:32:53,00001000000509846663,eup7IxzHZrNWdwBPzq2oDYaG7aFyz2kO9GO/qr9mVJD0t+...
1,4.0,A,1,2025-02-09 00:00:00,15.00,17.40,I,MXN,None,11590,...,601,RENJ0112285T3,JESUS REYNA NUÑEZ,S01,62577,605,7392dfdb-cc77-4d47-b3f2-a2840f2af633,2025-02-09T23:05:48,00001000000509846663,N6Pzryxiu+XDIQNIzhG8L1bg40hGfsp92qvAlbE9rxzjw6...
2,4.0,A,6,2025-08-05 00:00:00,50.00,50.00,I,MXN,None,01590,...,601,SAME030423JJ1,EMILIO SAID MACCISE,S01,52779,605,553856ff-c389-4002-b361-4d3f70aa8d32,2025-08-05T10:40:25,00001000000509846663,ICDSyis7GQzvoJTY7j09Ky3rXOGM7f8IZZ+r4AzDSu0y7z...
3,4.0,A,187,2025-04-28 19:01:22,390.50,452.98,I,MXN,None,66197,...,601,GAGJ010522MN7,JAVIER GARZA GUERRA,CP01,66266,605,36d47f4f-f779-4429-ba3d-e006efe72a15,2025-04-28T19:01:22,00001000000509846663,hpjVV/7xblIGKfr+ijNufD+S7gImsSQvWjW2JY1g0TAQy8...
4,4.0,NaN,19,2025-02-24 10:04:51,12.00,12.00,I,MXN,None,66197,...,601,RENJ0112285T3,JESUS REYNA NUÑEZ,S01,62577,605,85b29015-630d-477e-9f34-ed5e878c60d5,2025-02-24T10:04:51,00001000000509846663,Na6QorWnAJGTgPRZxsJHXwVUPDdo64mlPmB483mgmVrZsX...


In [17]:
# Ejecuta todas las validaciones con rango anual completo.
all_checks = validate_all(comprobantes, concepts, start="2025-01-01", end="2025-12-31")
for name, df in all_checks.items():
    print(name, "rows:", len(df))
all_checks

duplicate_uuid rows: 0
tax_math rows: 0
date_range rows: 0


{'duplicate_uuid': Empty DataFrame
 Columns: [uuid, count]
 Index: [],
 'tax_math': Empty DataFrame
 Columns: [uuid, subtotal_reported, subtotal_calculated, difference]
 Index: [],
 'date_range': Empty DataFrame
 Columns: [version, series, folio_number, issue_date, subtotal, total, invoice_type, currency, exchange_rate, place_of_issue, issuer_rfc, issuer_name, issuer_tax_regime, receiver_rfc, receiver_name, receiver_cfdi_use, receiver_fiscal_address, receiver_tax_regime, uuid, stamp_date, sat_certificate_number, cfd_seal]
 Index: []
 
 [0 rows x 22 columns]}

## 5.1) Ejemplo con errores simulados (opcional)

Estas celdas crean copias de los datos para mostrar validaciones con hallazgos.

In [18]:
# Crea copias y provoca: UUID duplicado y error de subtotal.
import pandas as pd

demo_comprobantes = comprobantes.copy()
demo_concepts = concepts.copy()

if not demo_comprobantes.empty:
    duplicated_row = demo_comprobantes.iloc[[0]].copy()
    demo_comprobantes = pd.concat([demo_comprobantes, duplicated_row], ignore_index=True)
    demo_comprobantes.loc[0, "subtotal"] = str(float(demo_comprobantes.loc[0, "subtotal"]) + 123.45)

print("Demo invoices:", len(demo_comprobantes))
print("Demo concepts:", len(demo_concepts))

Demo invoices: 14
Demo concepts: 14


In [19]:
# Corre validaciones sobre datos simulados para ver filas con problemas.
print(check_duplicate_uuid(demo_comprobantes))
print(check_tax_math(demo_comprobantes, demo_concepts))

                                   uuid  count
0  e5bcaecc-3fcd-4380-a8c4-d9552d14281c      2
                                   uuid  subtotal_reported  \
0  e5bcaecc-3fcd-4380-a8c4-d9552d14281c             123.45   

   subtotal_calculated  difference  
0                  0.0      123.45  


## 6) Lectura individual y salida JSON

Puedes leer un XML individual y convertir todo el resultado a lista de JSONs por factura.

In [20]:
# Busca archivos XML para probar lectura individual.
xml_files = sorted(list(data_folder.glob("*.xml")) + list(data_folder.glob("*.XML")))

if xml_files:
    # Lee un solo XML con read_cfdi.
    one = read_cfdi(xml_files[0])
    print("Single file:", xml_files[0].name)
    print("Keys:", one.keys())

    # Convierte la salida a JSON anidado por factura.
    print("Invoices in JSON:", len(one.to_json()))
    one.to_json()[0]
else:
    print("No XML files found in", data_folder)

Single file: 3bc6cb87-8afd-4d82-8834-4ce31207183b.xml
Keys: dict_keys(['comprobantes', 'conceptos', 'impuestos'])
Invoices in JSON: 1


## 7) Opcional: guardar JSON en archivo

Descomenta para exportar el resultado anidado.

In [21]:
# Importa json para exportar resultado anidado a archivo.
# import json

# Define el archivo de salida.
# output_path = Path("cfdi_output.json")

# Guarda invoices en formato JSON legible.
# output_path.write_text(json.dumps(data.to_json(), ensure_ascii=False, indent=2), encoding="utf-8")

# Muestra la ruta de salida creada.
# output_path